# Basketball Shot Detection & Tracking - Ilias

Builds the next basketball computer-vision phase after player detection/tracking and court vision mapping.

This notebook consumes the remapped 6-class basketball detector from Phase 1, optionally consumes the CVM court-keypoint model, and produces shot events, an annotated video, structured exports, and validation plots.

**Pipeline**

1. Environment + SageMaker path setup
2. Load player/object detector and optional CVM model
3. Generate or recover per-frame detections
4. Build smoothed ball and rim trajectories
5. Detect candidate shot attempts near the rim
6. Classify each attempt as `make`, `miss`, or `unknown`
7. Assign the likely shooter/team when evidence is available
8. Optionally project shot locations onto the top-down court
9. Export JSONL/CSV, annotated video, shot chart, and debug artifacts

## 1. Environment

Run once per kernel session. The package choices mirror the player tracking and CVM notebooks, with `pandas`, `matplotlib`, and `scipy` added for event tables and trajectory smoothing.


In [1]:
%pip install -q --upgrade ultralytics
%pip install -q supervision==0.27.0
%pip install -q opencv-python-headless pyyaml tqdm pandas imageio-ffmpeg matplotlib scipy
%pip install --no-cache-dir -q "git+https://github.com/roboflow/sports.git@feat/basketball"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
sagemaker-serve 1.10.1 requires onnxruntime, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.6 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import math
import os
import shutil
import subprocess
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import imageio_ffmpeg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import ultralytics
from tqdm import tqdm
from ultralytics import YOLO

try:
    from scipy.signal import savgol_filter
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    savgol_filter = None
    print(f"scipy smoothing unavailable; falling back to rolling medians: {exc}")

try:
    from sports.basketball import (
        CourtConfiguration,
        League,
        draw_court,
        draw_points_on_court,
    )
    from sports.basketball.config import MeasurementUnit
    from sports.common.team import TeamClassifier
    from sports.common.view import ViewTransformer
    SPORTS_AVAILABLE = True
except Exception as exc:
    SPORTS_AVAILABLE = False
    CourtConfiguration = League = MeasurementUnit = None
    TeamClassifier = ViewTransformer = None
    draw_court = draw_points_on_court = None
    print(f"sports package unavailable; court projection will be skipped: {exc}")

SEED = 45
np.random.seed(SEED)

HOME = Path.cwd()
RUNS_DIR = HOME / "runs_ilias"
SHOT_RUN_DIR = RUNS_DIR / "shot_tracking"
DETECTION_DIR = RUNS_DIR / "detections"
VIDEO_OUT_DIR = RUNS_DIR / "videos"
COMPRESSED_OUT_DIR = RUNS_DIR / "compressed"
EXPORT_DIR = RUNS_DIR / "exports"
CHART_DIR = RUNS_DIR / "charts"
DEBUG_DIR = RUNS_DIR / "debug_frames"
LOG_DIR = RUNS_DIR / "logs"

for folder in [
    RUNS_DIR,
    SHOT_RUN_DIR,
    DETECTION_DIR,
    VIDEO_OUT_DIR,
    COMPRESSED_OUT_DIR,
    EXPORT_DIR,
    CHART_DIR,
    DEBUG_DIR,
    LOG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

print("Current working directory:", HOME)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("ultralytics version:", ultralytics.__version__)
print("opencv version:", cv2.__version__)
print("output directory:", RUNS_DIR.resolve())
print("ffmpeg binary:", FFMPEG)


WARNING ⚠️ torchvision==0.24 is incompatible with torch==2.8.
Run 'pip install torchvision==0.23' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation


E0000 00:00:1779940092.774017    3240 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779940092.785715    3240 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779940092.953322    3240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779940092.953349    3240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779940092.953351    3240 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779940092.953353    3240 computation_placer.cc:177] computation placer already registered. Please check linka

Current working directory: /home/sagemaker-user/DLCNN_A3
CUDA available: True
GPU name: Tesla T4
ultralytics version: 8.4.56
opencv version: 4.13.0
output directory: /home/sagemaker-user/DLCNN_A3/runs_ilias
ffmpeg binary: /opt/conda/lib/python3.12/site-packages/imageio_ffmpeg/binaries/ffmpeg-linux-x86_64-v7.0.2


## 2. Paths and configuration

All generated artifacts go under `runs_ilias`. The source-video discovery prefers the broadcast clip used by the player tracking notebook, then falls back to the CVM sample video if that is the only local video present.

In [3]:
# Detector inference settings from Phase 1.
CONFIDENCE = 0.25
IOU = 0.50
IMGSZ = 960

# Remapped 6-class basketball schema from the player detection notebook.
CLASS_BALL = 0
CLASS_BALL_IN_BASKET = 1
CLASS_NUMBER = 2
CLASS_PLAYER = 3
CLASS_REFEREE = 4
CLASS_RIM = 5

CLASS_NAMES = {
    0: "ball",
    1: "ball-in-basket",
    2: "number",
    3: "player",
    4: "referee",
    5: "rim",
}
TRACKED_CLASSES = [CLASS_PLAYER, CLASS_REFEREE]

# Ball/rim trajectory settings. These are intentionally exposed because they are
# the first knobs to tune on a new broadcast angle.
MAX_BALL_GAP = 8                  # interpolate only short ball occlusions
MAX_RIM_GAP = 45                  # rim is static, so longer interpolation is safe
MAX_BALL_SPEED_PX_PER_FRAME = 95  # rejects single-frame false positives
BALL_SMOOTH_WINDOW = 7            # odd window for Savitzky-Golay / median smoothing
RIM_SMOOTH_WINDOW = 15

# Shot-candidate heuristics.
SHOT_WINDOW_FRAMES = 96           # temporal window inspected around a rim approach
MIN_SHOT_FRAMES = 10              # minimum usable ball samples in a candidate window
RIM_PROXIMITY_PX = 110            # ball must approach this close to the rim center
RELEASE_TO_RIM_MAX_FRAMES = 90    # search window before rim approach for release
SHOT_COOLDOWN_FRAMES = 45         # suppress duplicate detections of the same attempt
MIN_UPWARD_MOTION_PX = 22         # image y decreases when the ball moves upward
MIN_DOWNWARD_MOTION_PX = 18       # image y increases when the ball descends
HOOP_ZONE_WIDTH_PX = 76
HOOP_ZONE_HEIGHT_PX = 52

# Shooter/team settings.
SHOOTER_LOOKBACK_FRAMES = 18
MAX_SHOOTER_DISTANCE_PX = 170
ENABLE_TEAM_CLASSIFIER = SPORTS_AVAILABLE
TEAM_FIT_STRIDE = 30
MAX_TEAM_FIT_CROPS = 700
MIN_TEAM_FIT_CROPS = 20
TEAM_PALETTE = [sv.Color.from_hex("#2563eb"), sv.Color.from_hex("#ea580c")]

# Optional CVM projection settings.
CVM_CONF = 0.30
CVM_ANCHOR_CONF = 0.50
COURT_W, COURT_H = 94, 50
COURT_MARGIN = 2

SOURCE_VIDEO_DIR = HOME / "source"
SOURCE_VIDEO_CANDIDATES = [
    SOURCE_VIDEO_DIR / "boston-celtics-new-york-knicks-game-1-q1-04.28-04.20.mp4",
    SOURCE_VIDEO_DIR / "boston-celtics-new-york-knicks-game-1-q1-01.54-01.48.mp4",
    HOME / "Data" / "object-tracking.mp4",
]

SOURCE_VIDEO_PATH = next((p for p in SOURCE_VIDEO_CANDIDATES if p.exists()), SOURCE_VIDEO_CANDIDATES[0])
assert SOURCE_VIDEO_PATH.exists(), (
    "Source video not found. Expected one of:\n  "
    + "\n  ".join(str(p) for p in SOURCE_VIDEO_CANDIDATES)
    + "\nPlace the video in SageMaker and update SOURCE_VIDEO_PATH if needed."
)

VIDEO_STEM = SOURCE_VIDEO_PATH.stem
DETECTIONS_JSONL = DETECTION_DIR / f"{VIDEO_STEM}-detections.jsonl"
TARGET_VIDEO_PATH = VIDEO_OUT_DIR / f"{VIDEO_STEM}-shots.mp4"
TARGET_VIDEO_COMPRESSED_PATH = COMPRESSED_OUT_DIR / f"{VIDEO_STEM}-shots-compressed.mp4"
SHOT_EVENTS_JSONL = EXPORT_DIR / f"{VIDEO_STEM}-shot-events.jsonl"
SHOT_EVENTS_CSV = EXPORT_DIR / f"{VIDEO_STEM}-shot-events.csv"
SHOT_CHART_PATH = CHART_DIR / f"{VIDEO_STEM}-shot-chart.png"
BALL_TRAJECTORY_PLOT = DEBUG_DIR / f"{VIDEO_STEM}-ball-trajectory.png"
BALL_RIM_DISTANCE_PLOT = DEBUG_DIR / f"{VIDEO_STEM}-ball-to-rim-distance.png"


In [4]:
def count_decodable_frames(p: Path) -> int:
    """Count frames OpenCV can actually decode, not just metadata."""
    cap = cv2.VideoCapture(str(p))
    n = 0
    while True:
        ok, _ = cap.read()
        if not ok:
            break
        n += 1
    cap.release()
    return n


def video_metadata(p: Path) -> dict:
    cap = cv2.VideoCapture(str(p))
    meta = {
        "frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    cap.release()
    return meta


SOURCE_META = video_metadata(SOURCE_VIDEO_PATH)
SOURCE_FRAME_COUNT = count_decodable_frames(SOURCE_VIDEO_PATH)
SOURCE_FPS = SOURCE_META["fps"] if SOURCE_META["fps"] and SOURCE_META["fps"] > 0 else 30.0

print(f"Source: {SOURCE_VIDEO_PATH}")
print(f"  Metadata frame count: {SOURCE_META['frames']}")
print(f"  Actually decodable:   {SOURCE_FRAME_COUNT}")
print(f"  FPS:                  {SOURCE_FPS:.3f}")
print(f"  Size:                 {SOURCE_META['width']}x{SOURCE_META['height']}")

assert SOURCE_FRAME_COUNT > 0, f"No decodable frames found in {SOURCE_VIDEO_PATH}"
if SOURCE_META["frames"] and SOURCE_FRAME_COUNT < SOURCE_META["frames"] * 0.95:
    warnings.warn(
        f"Source may be truncated: {SOURCE_FRAME_COUNT}/{SOURCE_META['frames']} frames decode. "
        "The pipeline will still run on decodable frames."
    )

print("\nOutput paths:")
for p in [DETECTIONS_JSONL, TARGET_VIDEO_PATH, TARGET_VIDEO_COMPRESSED_PATH, SHOT_EVENTS_JSONL, SHOT_EVENTS_CSV, SHOT_CHART_PATH]:
    print(" ", p)


Source: /home/sagemaker-user/DLCNN_A3/source/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20.mp4
  Metadata frame count: 238
  Actually decodable:   238
  FPS:                  30.000
  Size:                 1920x1080

Output paths:
  /home/sagemaker-user/DLCNN_A3/runs_ilias/detections/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-detections.jsonl
  /home/sagemaker-user/DLCNN_A3/runs_ilias/videos/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shots.mp4
  /home/sagemaker-user/DLCNN_A3/runs_ilias/compressed/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shots-compressed.mp4
  /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.jsonl
  /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.csv
  /home/sagemaker-user/DLCNN_A3/runs_ilias/charts/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-chart.png


## 3. Load detector and optional CVM model

The player/object detector is required. We first try the checkpoint path used in Phase 1, then search existing project folders for `best.pt` files.

The CVM pose model is optional. If it is missing, shot detection/export/video rendering still run, and court-coordinate fields stay empty.


In [5]:
def score_detector_weight(path: Path) -> tuple:
    text = str(path).lower()
    score = 0
    if path.name == "best.pt":
        score += 20
    for token in ["basketball", "yolo26s", "player", "100ep", "map_best"]:
        if token in text:
            score += 25
    for token in ["pose", "cvm", "court", "keypoint"]:
        if token in text:
            score -= 80
    try:
        mtime = path.stat().st_mtime
        size = path.stat().st_size
    except FileNotFoundError:
        mtime = 0
        size = 0
    return (score, mtime, size)


def discover_detector_weights(home: Path) -> list[Path]:
    preferred = home / "my_checkpoints" / "yolo26s_basketball_100ep_mAP_best.pt"
    candidates = []
    if preferred.exists():
        candidates.append(preferred)

    for root in [home / "runs_ilias", home / "runs_abdo", home / "runs", home / "my_checkpoints"]:
        if not root.exists():
            continue
        candidates.extend(root.rglob("weights/best.pt"))
        candidates.extend(root.glob("*.pt"))

    # Deduplicate while preserving Path objects.
    candidates = sorted(set(candidates), key=score_detector_weight, reverse=True)
    return candidates


def discover_cvm_weights(home: Path) -> list[Path]:
    preferred = home / "runs" / "pose" / "runs" / "CVM" / "weights" / "best.pt"
    candidates = []
    if preferred.exists():
        candidates.append(preferred)
    for root in [home / "runs_ilias", home / "runs", home / "runs_abdo", home / "my_checkpoints"]:
        if not root.exists():
            continue
        for p in root.rglob("weights/best.pt"):
            text = str(p).lower()
            if "cvm" in text or "pose" in text or "court" in text:
                candidates.append(p)
    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates


weight_candidates = discover_detector_weights(HOME)
print("Detector weight candidates:")
for p in weight_candidates[:12]:
    score, mtime, size = score_detector_weight(p)
    print(f"  score={score:>4}  size={size/1e6:>6.1f} MB  {p}")

if not weight_candidates:
    raise FileNotFoundError(
        "No detector weights found. Place the trained 6-class detector at\n"
        f"  {HOME / 'my_checkpoints' / 'yolo26s_basketball_100ep_mAP_best.pt'}\n"
        "or under one of: runs_ilias, runs_abdo, runs, my_checkpoints."
    )

BEST_PT = weight_candidates[0]
detector = YOLO(str(BEST_PT))
print(f"\nUsing detector weights: {BEST_PT}")
print(f"Detector task: {getattr(detector, 'task', 'unknown')}")
print(f"Detector names: {detector.names}")

# Optional CVM model.
CVM_MODEL_PATH = None
CVM_model = None
if SPORTS_AVAILABLE:
    cvm_candidates = discover_cvm_weights(HOME)
    print("\nCVM weight candidates:")
    for p in cvm_candidates[:8]:
        print(f"  size={p.stat().st_size/1e6:>6.1f} MB  {p}")
    if cvm_candidates:
        CVM_MODEL_PATH = cvm_candidates[0]
        CVM_model = YOLO(str(CVM_MODEL_PATH))
        print(f"Using CVM weights: {CVM_MODEL_PATH}")
    else:
        print("No CVM weights found; court projection will be skipped.")
else:
    print("sports package not available; court projection will be skipped.")


Detector weight candidates:
  score= 100  size=  20.3 MB  /home/sagemaker-user/DLCNN_A3/my_checkpoints/yolo26s_basketball_100ep_mAP_best.pt
  score=  70  size=  20.3 MB  /home/sagemaker-user/DLCNN_A3/runs_abdo/basketball/yolo26s_960/weights/best.pt
  score=  20  size=  20.3 MB  /home/sagemaker-user/DLCNN_A3/runs/mlflow/950406735019881577/8f963b873f9044ff9d0686011fddbda6/artifacts/weights/best.pt
  score=  20  size=  11.5 MB  /home/sagemaker-user/DLCNN_A3/runs/mlflow/148027301727702339/f944372ebfb143e09db8dfbd9705986e/artifacts/weights/best.pt
  score=-140  size=  11.5 MB  /home/sagemaker-user/DLCNN_A3/runs/pose/runs/CVM/weights/best.pt

Using detector weights: /home/sagemaker-user/DLCNN_A3/my_checkpoints/yolo26s_basketball_100ep_mAP_best.pt
Detector task: detect
Detector names: {0: 'ball', 1: 'ball-in-basket', 2: 'number', 3: 'player', 4: 'referee', 5: 'rim'}

CVM weight candidates:
  size=  11.5 MB  /home/sagemaker-user/DLCNN_A3/runs/pose/runs/CVM/weights/best.pt


Using CVM weights: /home/sagemaker-user/DLCNN_A3/runs/pose/runs/CVM/weights/best.pt


## 4. Generate or load detections

Mode A loads a matching JSONL detection log if one exists. The search checks `runs_ilias/detections` first, then older project output folders only as read-only sources.

Mode B runs YOLO over the source video, tracks players/referees with ByteTrack, and writes a fresh log to `runs_ilias/detections/<video-stem>-detections.jsonl`.

The ball is **not** ByteTracked. Fast, tiny ball detections are kept as per-frame observations and handled by the custom trajectory smoother below.


In [6]:
FORCE_REGENERATE_DETECTIONS = False


def load_detections_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_detection_record(f, frame_idx: int, det: dict):
    f.write(json.dumps(det) + "\n")


def find_existing_detection_log(video_stem: str) -> Path | None:
    exact = DETECTION_DIR / f"{video_stem}-detections.jsonl"
    if exact.exists():
        return exact

    candidates = []
    for p in [HOME / f"{video_stem}-detections.jsonl"]:
        if p.exists():
            candidates.append(p)

    for root in [HOME / "runs_abdo", HOME / "runs", HOME / "runs_ilias"]:
        if not root.exists():
            continue
        candidates.extend(root.rglob(f"{video_stem}-detections.jsonl"))

    candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0] if candidates else None


def detection_name(result, cid: int) -> str:
    names = getattr(result, "names", {}) or {}
    if isinstance(names, dict):
        return str(names.get(int(cid), CLASS_NAMES.get(int(cid), str(cid))))
    if int(cid) < len(names):
        return str(names[int(cid)])
    return CLASS_NAMES.get(int(cid), str(cid))


def detections_to_json_records(det_set: sv.Detections, tracker_ids, result, frame_idx: int) -> list[dict]:
    rows = []
    if len(det_set) == 0:
        return rows
    if tracker_ids is None:
        tracker_ids = [None] * len(det_set)
    for (x1, y1, x2, y2), cid, conf, tid in zip(
        det_set.xyxy,
        det_set.class_id,
        det_set.confidence,
        tracker_ids,
    ):
        cid = int(cid)
        rows.append({
            "frame": int(frame_idx),
            "track_id": int(tid) if tid is not None and not pd.isna(tid) else None,
            "class": detection_name(result, cid),
            "class_id": cid,
            "bbox_xyxy": [float(x1), float(y1), float(x2), float(y2)],
            "conf": float(conf),
        })
    return rows


def generate_detections_jsonl(source_video_path: Path, output_jsonl: Path) -> Path:
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    video_info = sv.VideoInfo.from_video_path(str(source_video_path))
    tracker = sv.ByteTrack(frame_rate=int(round(video_info.fps or SOURCE_FPS or 30)))

    cap = cv2.VideoCapture(str(source_video_path))
    n_records = 0
    frame_idx = 0

    with output_jsonl.open("w") as f:
        progress = tqdm(total=SOURCE_FRAME_COUNT, desc="running detector")
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            result = detector.predict(
                frame,
                conf=CONFIDENCE,
                iou=IOU,
                imgsz=IMGSZ,
                verbose=False,
                project=str(RUNS_DIR),
                name="shot_detection_predict",
                exist_ok=True,
            )[0]
            detections = sv.Detections.from_ultralytics(result)

            if len(detections):
                tracked_mask = np.isin(detections.class_id, TRACKED_CLASSES)
                tracked = tracker.update_with_detections(detections[tracked_mask])
                other = detections[~tracked_mask]

                for row in detections_to_json_records(tracked, tracked.tracker_id, result, frame_idx):
                    write_detection_record(f, frame_idx, row)
                    n_records += 1
                for row in detections_to_json_records(other, [None] * len(other), result, frame_idx):
                    write_detection_record(f, frame_idx, row)
                    n_records += 1

            frame_idx += 1
            progress.update(1)
        progress.close()

    cap.release()
    print(f"Wrote {n_records:,} detection records to {output_jsonl}")
    return output_jsonl


existing_detection_log = None if FORCE_REGENERATE_DETECTIONS else find_existing_detection_log(VIDEO_STEM)

if existing_detection_log is not None and existing_detection_log.exists():
    if existing_detection_log.resolve() != DETECTIONS_JSONL.resolve():
        shutil.copy2(existing_detection_log, DETECTIONS_JSONL)
        print(f"Copied existing detection log into runs_ilias: {existing_detection_log} -> {DETECTIONS_JSONL}")
    else:
        print(f"Using existing detection log: {DETECTIONS_JSONL}")
else:
    print("No existing detection log found; generating detections now.")
    generate_detections_jsonl(SOURCE_VIDEO_PATH, DETECTIONS_JSONL)

assert DETECTIONS_JSONL.exists(), f"Detection log was not created: {DETECTIONS_JSONL}"
detection_records = load_detections_jsonl(DETECTIONS_JSONL)
print(f"Loaded {len(detection_records):,} detection records")
print("Class counts:", dict(Counter(r.get("class") for r in detection_records)))


Copied existing detection log into runs_ilias: /home/sagemaker-user/DLCNN_A3/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-detections.jsonl -> /home/sagemaker-user/DLCNN_A3/runs_ilias/detections/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-detections.jsonl
Loaded 5,081 detection records
Class counts: {'player': 2369, 'referee': 699, 'number': 1623, 'rim': 251, 'ball': 122, 'ball-in-basket': 17}


## 5. Extract ball, rim, player, and ball-in-basket observations

The JSONL log is converted into frame-indexed observations.

In [7]:
def group_detections_by_frame(records: list[dict]) -> dict[int, list[dict]]:
    grouped = defaultdict(list)
    for row in records:
        grouped[int(row["frame"])].append(row)
    return grouped


def bbox_center(bbox) -> tuple[float, float]:
    x1, y1, x2, y2 = map(float, bbox)
    return (0.5 * (x1 + x2), 0.5 * (y1 + y2))


def bbox_bottom_center(bbox) -> tuple[float, float]:
    x1, y1, x2, y2 = map(float, bbox)
    return (0.5 * (x1 + x2), y2)


def bbox_upper_body_center(bbox) -> tuple[float, float]:
    x1, y1, x2, y2 = map(float, bbox)
    return (0.5 * (x1 + x2), y1 + 0.35 * (y2 - y1))


def bbox_area(bbox) -> float:
    x1, y1, x2, y2 = map(float, bbox)
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_wh(bbox) -> tuple[float, float]:
    x1, y1, x2, y2 = map(float, bbox)
    return max(0.0, x2 - x1), max(0.0, y2 - y1)


def frame_class_counts(detections_by_frame: dict[int, list[dict]]) -> pd.DataFrame:
    rows = []
    for frame, dets in sorted(detections_by_frame.items()):
        counts = Counter(int(d["class_id"]) for d in dets)
        rows.append({
            "frame": frame,
            "ball": counts.get(CLASS_BALL, 0),
            "ball_in_basket": counts.get(CLASS_BALL_IN_BASKET, 0),
            "rim": counts.get(CLASS_RIM, 0),
            "player": counts.get(CLASS_PLAYER, 0),
            "referee": counts.get(CLASS_REFEREE, 0),
            "number": counts.get(CLASS_NUMBER, 0),
        })
    return pd.DataFrame(rows)


detections_by_frame = group_detections_by_frame(detection_records)
max_logged_frame = max(detections_by_frame.keys()) if detections_by_frame else -1
FRAME_COUNT = max(SOURCE_FRAME_COUNT, max_logged_frame + 1)
counts_df = frame_class_counts(detections_by_frame)

print(f"Frames in source: {SOURCE_FRAME_COUNT}")
print(f"Frames covered by detections: {max_logged_frame + 1}")
print(f"Pipeline frame count: {FRAME_COUNT}")
if len(counts_df):
    display(counts_df[["ball", "ball_in_basket", "rim", "player", "referee"]].sum().to_frame("detections"))
else:
    print("No detections found. The downstream pipeline will produce empty outputs.")


Frames in source: 238
Frames covered by detections: 238
Pipeline frame count: 238


,detections
ball,122
ball_in_basket,17
rim,251
player,2369
referee,699


## 6. Build a smoothed ball trajectory

The ball is small, fast, and often hidden behind players. This stage chooses one likely ball observation per frame, rejects obvious jumps, interpolates only short gaps, and applies light smoothing. Rim detections get their own stable track because the rim is the anchor for make/miss classification.


In [8]:
def euclidean(a, b) -> float:
    if a is None or b is None:
        return float("inf")
    return float(np.linalg.norm(np.array(a, dtype=float) - np.array(b, dtype=float)))


def select_best_ball_detection(frame_dets: list[dict], prev_xy=None) -> dict | None:
    candidates = [d for d in frame_dets if int(d["class_id"]) in [CLASS_BALL, CLASS_BALL_IN_BASKET]]
    if not candidates:
        return None

    scored = []
    for d in candidates:
        center = bbox_center(d["bbox_xyxy"])
        w, h = bbox_wh(d["bbox_xyxy"])
        area = bbox_area(d["bbox_xyxy"])
        class_bonus = 0.10 if int(d["class_id"]) == CLASS_BALL else 0.02
        shape_penalty = 0.0
        if w > 0 and h > 0:
            aspect = max(w / h, h / w)
            shape_penalty = max(0.0, aspect - 2.5) * 0.05
        area_penalty = max(0.0, area - 4500.0) / 25000.0
        temporal_penalty = 0.0 if prev_xy is None else min(euclidean(center, prev_xy) / 300.0, 1.0)
        score = float(d.get("conf", 0.0)) + class_bonus - shape_penalty - area_penalty - temporal_penalty
        scored.append((score, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def select_best_rim_detection(frame_dets: list[dict], prev_xy=None) -> dict | None:
    candidates = [d for d in frame_dets if int(d["class_id"]) == CLASS_RIM]
    if not candidates:
        return None
    scored = []
    for d in candidates:
        center = bbox_center(d["bbox_xyxy"])
        temporal_penalty = 0.0 if prev_xy is None else min(euclidean(center, prev_xy) / 250.0, 1.0)
        score = float(d.get("conf", 0.0)) - temporal_penalty
        scored.append((score, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def contiguous_runs(indices: np.ndarray) -> list[np.ndarray]:
    if len(indices) == 0:
        return []
    splits = np.where(np.diff(indices) > 1)[0] + 1
    return [run for run in np.split(indices, splits) if len(run)]


def smooth_values(values: np.ndarray, window: int, poly: int = 2) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    out = values.copy()
    valid_idx = np.flatnonzero(~np.isnan(values))
    for run in contiguous_runs(valid_idx):
        if len(run) < 3:
            continue
        run_values = values[run]
        if SCIPY_AVAILABLE and len(run) >= 5:
            w = min(window, len(run) if len(run) % 2 == 1 else len(run) - 1)
            w = max(5, w)
            if w % 2 == 0:
                w -= 1
            if w >= 5 and w > poly:
                out[run] = savgol_filter(run_values, window_length=w, polyorder=min(poly, w - 2), mode="interp")
                continue
        out[run] = pd.Series(run_values).rolling(3, center=True, min_periods=1).median().to_numpy()
    return out


def interpolate_short_gaps(df: pd.DataFrame, x_col: str, y_col: str, max_gap: int, flag_col: str) -> pd.DataFrame:
    df = df.copy()
    valid = df[x_col].notna() & df[y_col].notna()
    valid_frames = df.loc[valid, "frame"].to_numpy(dtype=int)
    for left_frame, right_frame in zip(valid_frames[:-1], valid_frames[1:]):
        gap = right_frame - left_frame - 1
        if gap <= 0 or gap > max_gap:
            continue
        left = df.loc[df["frame"] == left_frame].iloc[0]
        right = df.loc[df["frame"] == right_frame].iloc[0]
        for frame in range(left_frame + 1, right_frame):
            alpha = (frame - left_frame) / (right_frame - left_frame)
            df.loc[df["frame"] == frame, x_col] = (1 - alpha) * left[x_col] + alpha * right[x_col]
            df.loc[df["frame"] == frame, y_col] = (1 - alpha) * left[y_col] + alpha * right[y_col]
            df.loc[df["frame"] == frame, flag_col] = True
            if "conf" in df.columns:
                df.loc[df["frame"] == frame, "conf"] = 0.0
            if "rim_conf" in df.columns:
                df.loc[df["frame"] == frame, "rim_conf"] = 0.0
    return df


def build_ball_track(detections_by_frame: dict[int, list[dict]], frame_count: int) -> pd.DataFrame:
    rows = []
    prev_xy = None
    for frame in range(frame_count):
        det = select_best_ball_detection(detections_by_frame.get(frame, []), prev_xy=prev_xy)
        if det is None:
            rows.append({
                "frame": frame,
                "x": np.nan,
                "y": np.nan,
                "raw_x": np.nan,
                "raw_y": np.nan,
                "conf": np.nan,
                "bbox": None,
                "area": np.nan,
                "is_interpolated": False,
                "is_outlier": False,
                "source_class_id": None,
            })
            continue
        x, y = bbox_center(det["bbox_xyxy"])
        prev_xy = (x, y)
        rows.append({
            "frame": frame,
            "x": x,
            "y": y,
            "raw_x": x,
            "raw_y": y,
            "conf": float(det.get("conf", 0.0)),
            "bbox": det["bbox_xyxy"],
            "area": bbox_area(det["bbox_xyxy"]),
            "is_interpolated": False,
            "is_outlier": False,
            "source_class_id": int(det["class_id"]),
        })

    df = pd.DataFrame(rows)

    # Reject single-frame jumps from the last accepted observation. The threshold
    # is high enough to allow real shot arcs but filters common false positives.
    last_frame = None
    last_xy = None
    for idx, row in df[df["x"].notna()].iterrows():
        xy = (row["x"], row["y"])
        if last_xy is not None:
            gap = max(1, int(row["frame"] - last_frame))
            speed = euclidean(xy, last_xy) / gap
            if speed > MAX_BALL_SPEED_PX_PER_FRAME:
                df.loc[idx, ["x", "y"]] = np.nan
                df.loc[idx, "is_outlier"] = True
                continue
        last_frame = int(row["frame"])
        last_xy = xy

    df = interpolate_short_gaps(df, "x", "y", MAX_BALL_GAP, "is_interpolated")
    df["smooth_x"] = smooth_values(df["x"].to_numpy(dtype=float), BALL_SMOOTH_WINDOW)
    df["smooth_y"] = smooth_values(df["y"].to_numpy(dtype=float), BALL_SMOOTH_WINDOW)
    df["x"] = df["smooth_x"]
    df["y"] = df["smooth_y"]
    return df


def build_rim_track(detections_by_frame: dict[int, list[dict]], frame_count: int) -> pd.DataFrame:
    rows = []
    prev_xy = None
    for frame in range(frame_count):
        det = select_best_rim_detection(detections_by_frame.get(frame, []), prev_xy=prev_xy)
        if det is None:
            rows.append({
                "frame": frame,
                "rim_x": np.nan,
                "rim_y": np.nan,
                "rim_bbox": None,
                "rim_conf": np.nan,
                "rim_w": np.nan,
                "rim_h": np.nan,
                "is_interpolated": False,
            })
            continue
        x, y = bbox_center(det["bbox_xyxy"])
        w, h = bbox_wh(det["bbox_xyxy"])
        prev_xy = (x, y)
        rows.append({
            "frame": frame,
            "rim_x": x,
            "rim_y": y,
            "rim_bbox": det["bbox_xyxy"],
            "rim_conf": float(det.get("conf", 0.0)),
            "rim_w": w,
            "rim_h": h,
            "is_interpolated": False,
        })

    df = pd.DataFrame(rows)
    df = interpolate_short_gaps(df, "rim_x", "rim_y", MAX_RIM_GAP, "is_interpolated")
    df["rim_x"] = smooth_values(df["rim_x"].to_numpy(dtype=float), RIM_SMOOTH_WINDOW)
    df["rim_y"] = smooth_values(df["rim_y"].to_numpy(dtype=float), RIM_SMOOTH_WINDOW)
    return df


ball_track = build_ball_track(detections_by_frame, FRAME_COUNT)
rim_track = build_rim_track(detections_by_frame, FRAME_COUNT)

print("Ball trajectory:")
print(f"  raw/interpolated usable frames: {int(ball_track['x'].notna().sum())}/{FRAME_COUNT}")
print(f"  interpolated frames:            {int(ball_track['is_interpolated'].sum())}")
print(f"  rejected outliers:              {int(ball_track['is_outlier'].sum())}")
print("Rim trajectory:")
print(f"  usable frames:                  {int(rim_track['rim_x'].notna().sum())}/{FRAME_COUNT}")
print(f"  interpolated frames:            {int(rim_track['is_interpolated'].sum())}")

display(ball_track.head())


Ball trajectory:
  raw/interpolated usable frames: 123/238
  interpolated frames:            6
  rejected outliers:              0
Rim trajectory:
  usable frames:                  238/238
  interpolated frames:            0


,frame,x,y,raw_x,raw_y,conf,bbox,area,is_interpolated,is_outlier,source_class_id,smooth_x,smooth_y
0,0,602.945135,578.001046,602.207550,578.215973,0.833680,"[584.7681274414062, 562.5841064453125, 619.646...",1090.442901,False,False,0.0,602.945135,578.001046
1,1,607.675003,588.709163,609.211700,588.329590,0.877036,"[594.1715087890625, 572.92236328125, 624.25189...",926.910561,False,False,0.0,607.675003,588.709163
2,2,611.696912,601.310538,611.512329,601.159698,0.869679,"[596.9462280273438, 586.4086303710938, 626.078...",859.462196,False,False,0.0,611.696912,601.310538
3,3,615.010862,615.805171,613.720276,616.385864,0.829178,"[599.3245239257812, 601.5845947265625, 628.116...",852.301619,False,False,0.0,615.010862,615.805171
4,4,617.616851,632.193063,618.292908,631.927856,0.864499,"[605.5925903320312, 616.764892578125, 630.9932...",770.297814,False,False,0.0,617.616851,632.193063


## 7. Detect shot candidates

A candidate shot is a ball trajectory segment that approaches the rim and looks enough like a shot arc: upward motion before the rim approach, descent near/after the rim, enough observed/interpolated samples, and a cooldown to avoid double-counting.


In [9]:
def add_ball_rim_distance(ball_track: pd.DataFrame, rim_track: pd.DataFrame) -> pd.DataFrame:
    df = ball_track[["frame", "x", "y", "conf", "is_interpolated", "is_outlier"]].merge(
        rim_track[["frame", "rim_x", "rim_y", "rim_w", "rim_h", "rim_conf"]],
        on="frame",
        how="left",
    )
    valid = df[["x", "y", "rim_x", "rim_y"]].notna().all(axis=1)
    df["rim_distance_px"] = np.nan
    df.loc[valid, "rim_distance_px"] = np.sqrt(
        (df.loc[valid, "x"] - df.loc[valid, "rim_x"]) ** 2
        + (df.loc[valid, "y"] - df.loc[valid, "rim_y"]) ** 2
    )
    return df


def cluster_frames(frames: list[int], max_gap: int = 5) -> list[list[int]]:
    frames = sorted(set(int(f) for f in frames))
    if not frames:
        return []
    clusters = [[frames[0]]]
    for f in frames[1:]:
        if f - clusters[-1][-1] <= max_gap:
            clusters[-1].append(f)
        else:
            clusters.append([f])
    return clusters


def estimate_release_frame(distance_df: pd.DataFrame, start_frame: int, rim_frame: int) -> int:
    pre = distance_df[
        (distance_df["frame"] >= start_frame)
        & (distance_df["frame"] <= rim_frame)
        & distance_df["x"].notna()
        & distance_df["y"].notna()
    ].copy()
    if len(pre) == 0:
        return int(start_frame)
    if len(pre) < 3:
        return int(pre.iloc[0]["frame"])

    frames = pre["frame"].to_numpy(dtype=int)
    y = pre["y"].to_numpy(dtype=float)
    dy = np.diff(y)
    upward_steps = np.flatnonzero(dy < -1.5)
    if len(upward_steps) == 0:
        return int(frames[max(0, len(frames) - 1 - min(RELEASE_TO_RIM_MAX_FRAMES, len(frames) - 1))])

    runs = contiguous_runs(upward_steps)
    # Use the latest sustained upward run before the rim approach.
    runs = sorted(runs, key=lambda r: (len(r), r[-1]), reverse=True)
    chosen = runs[0]
    release_idx = max(0, int(chosen[0]) - 1)
    return int(frames[release_idx])


def candidate_motion_stats(window_df: pd.DataFrame, rim_frame: int) -> dict:
    valid = window_df[window_df["x"].notna() & window_df["y"].notna()].copy()
    if len(valid) == 0:
        return {"upward_motion_px": 0.0, "downward_motion_px": 0.0, "valid_frames": 0}

    pre = valid[valid["frame"] <= rim_frame]
    post = valid[valid["frame"] >= rim_frame]
    upward_motion = 0.0
    downward_motion = 0.0
    if len(pre) >= 2:
        upward_motion = float(pre.iloc[0]["y"] - pre["y"].min())
    if len(post) >= 2:
        downward_motion = float(post["y"].max() - post.iloc[0]["y"])
    return {
        "upward_motion_px": upward_motion,
        "downward_motion_px": downward_motion,
        "valid_frames": int(len(valid)),
    }


def has_ball_in_basket_signal(detections_by_frame: dict[int, list[dict]], start_frame: int, end_frame: int) -> bool:
    for f in range(max(0, start_frame), min(FRAME_COUNT - 1, end_frame) + 1):
        if any(int(d["class_id"]) == CLASS_BALL_IN_BASKET for d in detections_by_frame.get(f, [])):
            return True
    return False


def detect_shot_candidates(
    ball_track: pd.DataFrame,
    rim_track: pd.DataFrame,
    detections_by_frame: dict[int, list[dict]],
) -> tuple[list[dict], pd.DataFrame]:
    distance_df = add_ball_rim_distance(ball_track, rim_track)
    near = distance_df[
        distance_df["rim_distance_px"].notna()
        & (distance_df["rim_distance_px"] <= RIM_PROXIMITY_PX)
    ]
    clusters = cluster_frames(near["frame"].astype(int).tolist(), max_gap=5)

    candidates = []
    last_rim_frame = -SHOT_COOLDOWN_FRAMES
    for cluster in clusters:
        cluster_df = distance_df[distance_df["frame"].isin(cluster)]
        if len(cluster_df) == 0:
            continue
        rim_frame = int(cluster_df.sort_values("rim_distance_px").iloc[0]["frame"])
        if rim_frame - last_rim_frame < SHOT_COOLDOWN_FRAMES:
            continue

        start_frame = max(0, rim_frame - RELEASE_TO_RIM_MAX_FRAMES)
        end_frame = min(FRAME_COUNT - 1, rim_frame + SHOT_WINDOW_FRAMES // 2)
        window_df = distance_df[(distance_df["frame"] >= start_frame) & (distance_df["frame"] <= end_frame)]
        stats = candidate_motion_stats(window_df, rim_frame)
        bib_signal = has_ball_in_basket_signal(detections_by_frame, rim_frame - 12, rim_frame + 24)
        min_dist = float(window_df["rim_distance_px"].min()) if window_df["rim_distance_px"].notna().any() else np.nan

        enough_frames = stats["valid_frames"] >= MIN_SHOT_FRAMES
        arc_like = (
            stats["upward_motion_px"] >= MIN_UPWARD_MOTION_PX
            and stats["downward_motion_px"] >= MIN_DOWNWARD_MOTION_PX
        )
        very_close = np.isfinite(min_dist) and min_dist <= RIM_PROXIMITY_PX * 0.65

        if not enough_frames:
            continue
        if not (arc_like or bib_signal or very_close):
            continue

        release_frame = estimate_release_frame(distance_df, start_frame, rim_frame)
        valid_window = window_df[window_df["x"].notna() & window_df["y"].notna()]
        if len(valid_window):
            start_frame = int(valid_window.iloc[0]["frame"])
            end_frame = int(valid_window.iloc[-1]["frame"])

        candidates.append({
            "shot_id": len(candidates) + 1,
            "start_frame": int(start_frame),
            "release_frame": int(release_frame),
            "rim_frame": int(rim_frame),
            "end_frame": int(end_frame),
            "min_rim_distance_px": min_dist,
            "upward_motion_px": stats["upward_motion_px"],
            "downward_motion_px": stats["downward_motion_px"],
            "valid_ball_frames": stats["valid_frames"],
            "candidate_reason": "ball approached rim with plausible arc" if arc_like else (
                "ball-in-basket signal near rim" if bib_signal else "ball passed very close to rim"
            ),
        })
        last_rim_frame = rim_frame

    return candidates, distance_df


shot_candidates, distance_df = detect_shot_candidates(ball_track, rim_track, detections_by_frame)
print(f"Detected {len(shot_candidates)} shot candidates")
display(pd.DataFrame(shot_candidates))


Detected 1 shot candidates


,shot_id,start_frame,release_frame,rim_frame,end_frame,min_rim_distance_px,upward_motion_px,downward_motion_px,valid_ball_frames,candidate_reason
0,1,137,137,219,237,5.020771,670.838051,56.31852,68,ball approached rim with plausible arc


## 8. Classify makes/misses/unknown

Classification uses multiple weak signals rather than a single rule:

- `ball-in-basket` detections near the rim
- descending ball crossing through the hoop zone
- ball moving away after a close rim approach
- enough post-rim trajectory to support a miss

Weak evidence stays `unknown`.


In [10]:
def rim_row_at(frame: int) -> pd.Series | None:
    if frame < 0 or frame >= len(rim_track):
        return None
    row = rim_track.iloc[int(frame)]
    if pd.isna(row["rim_x"]) or pd.isna(row["rim_y"]):
        return None
    return row


def hoop_zone(frame: int) -> tuple[float, float, float, float] | None:
    row = rim_row_at(frame)
    if row is None:
        return None
    rim_w = float(row["rim_w"]) if pd.notna(row.get("rim_w", np.nan)) else HOOP_ZONE_WIDTH_PX
    rim_h = float(row["rim_h"]) if pd.notna(row.get("rim_h", np.nan)) else HOOP_ZONE_HEIGHT_PX
    width = max(HOOP_ZONE_WIDTH_PX, rim_w * 1.4)
    height = max(HOOP_ZONE_HEIGHT_PX, rim_h * 1.6)
    return (float(row["rim_x"]), float(row["rim_y"]), width, height)


def point_inside_hoop_zone(x: float, y: float, zone: tuple[float, float, float, float]) -> bool:
    rim_x, rim_y, width, height = zone
    return abs(x - rim_x) <= width / 2 and abs(y - rim_y) <= height / 2


def ball_in_basket_frames(candidate: dict) -> list[int]:
    frames = []
    start = max(0, candidate["rim_frame"] - 12)
    end = min(FRAME_COUNT - 1, candidate["rim_frame"] + 30)
    for f in range(start, end + 1):
        zone = hoop_zone(f)
        for d in detections_by_frame.get(f, []):
            if int(d["class_id"]) != CLASS_BALL_IN_BASKET:
                continue
            cx, cy = bbox_center(d["bbox_xyxy"])
            if zone is None or point_inside_hoop_zone(cx, cy, zone):
                frames.append(f)
                break
    return frames


def crossed_hoop_plane(candidate: dict) -> bool:
    start = max(0, candidate["rim_frame"] - 18)
    end = min(FRAME_COUNT - 1, candidate["rim_frame"] + 28)
    rows = ball_track[(ball_track["frame"] >= start) & (ball_track["frame"] <= end) & ball_track["x"].notna() & ball_track["y"].notna()]
    if len(rows) < 2:
        return False

    rows = rows.sort_values("frame")
    for (_, a), (_, b) in zip(rows.iloc[:-1].iterrows(), rows.iloc[1:].iterrows()):
        zone = hoop_zone(int(b["frame"])) or hoop_zone(candidate["rim_frame"])
        if zone is None:
            return False
        rim_x, rim_y, width, height = zone
        descending = float(b["y"]) > float(a["y"])
        crosses_y = float(a["y"]) <= rim_y - height * 0.20 and float(b["y"]) >= rim_y + height * 0.20
        if not (descending and crosses_y):
            continue
        denom = float(b["y"] - a["y"])
        alpha = 0.5 if abs(denom) < 1e-6 else (rim_y - float(a["y"])) / denom
        x_at_rim = float(a["x"]) + alpha * float(b["x"] - a["x"])
        if abs(x_at_rim - rim_x) <= width / 2:
            return True
    return False


def classify_shot_result(candidate: dict) -> dict:
    rim_frame = int(candidate["rim_frame"])
    rim = rim_row_at(rim_frame)
    if rim is None:
        return {
            "result": "unknown",
            "result_confidence": 0.20,
            "result_reason": "rim unavailable",
            "ball_in_basket_frames": [],
        }

    bib_frames = ball_in_basket_frames(candidate)
    if bib_frames:
        return {
            "result": "make",
            "result_confidence": 0.92,
            "result_reason": "ball-in-basket detection near rim",
            "ball_in_basket_frames": bib_frames,
        }

    if crossed_hoop_plane(candidate):
        return {
            "result": "make",
            "result_confidence": 0.78,
            "result_reason": "descending ball crossed hoop plane",
            "ball_in_basket_frames": [],
        }

    post = distance_df[
        (distance_df["frame"] > rim_frame)
        & (distance_df["frame"] <= min(FRAME_COUNT - 1, rim_frame + SHOT_WINDOW_FRAMES // 2))
        & distance_df["rim_distance_px"].notna()
        & distance_df["x"].notna()
        & distance_df["y"].notna()
    ].copy()

    if len(post) < 3:
        return {
            "result": "unknown",
            "result_confidence": 0.30,
            "result_reason": "insufficient post-rim trajectory",
            "ball_in_basket_frames": [],
        }

    zone = hoop_zone(rim_frame)
    last = post.tail(min(5, len(post)))
    last_dist = float(last["rim_distance_px"].mean())
    min_dist = float(candidate.get("min_rim_distance_px", np.nan))
    moved_away = np.isfinite(min_dist) and last_dist > min_dist + RIM_PROXIMITY_PX * 0.55
    below_rim = float(last["y"].median()) > float(rim["rim_y"]) + HOOP_ZONE_HEIGHT_PX * 0.45
    outside_zone = True
    if zone is not None:
        outside_zone = not any(point_inside_hoop_zone(float(r["x"]), float(r["y"]), zone) for _, r in last.iterrows())

    if moved_away and (below_rim or outside_zone):
        return {
            "result": "miss",
            "result_confidence": 0.64,
            "result_reason": "approached rim then moved away",
            "ball_in_basket_frames": [],
        }

    return {
        "result": "unknown",
        "result_confidence": 0.36,
        "result_reason": "shot close to rim but result unclear",
        "ball_in_basket_frames": [],
    }


for candidate in shot_candidates:
    candidate.update(classify_shot_result(candidate))

shot_events = shot_candidates
print("Classified shot candidates:")
display(pd.DataFrame(shot_events))


Classified shot candidates:


,shot_id,start_frame,release_frame,rim_frame,end_frame,min_rim_distance_px,upward_motion_px,downward_motion_px,valid_ball_frames,candidate_reason,result,result_confidence,result_reason,ball_in_basket_frames
0,1,137,137,219,237,5.020771,670.838051,56.31852,68,ball approached rim with plausible arc,make,0.92,ball-in-basket detection near rim,"[220, 221, 222, 223, 224, 225, 226, 227, 228, ..."


## 9. Assign likely shooter/team

Shooter assignment looks backward from the estimated release frame and chooses the tracked player closest to the ball, using an upper-body anchor rather than the player's feet. Team assignment reuses the CVM notebook's unsupervised `TeamClassifier` style when the `sports` package and enough crops are available; otherwise `team_id` stays empty.


In [11]:
def ball_xy_at(frame: int, max_nearest_gap: int = 3) -> tuple[float, float] | None:
    frame = int(frame)
    if 0 <= frame < len(ball_track):
        row = ball_track.iloc[frame]
        if pd.notna(row["x"]) and pd.notna(row["y"]):
            return (float(row["x"]), float(row["y"]))
    start = max(0, frame - max_nearest_gap)
    end = min(len(ball_track) - 1, frame + max_nearest_gap)
    nearby = ball_track[(ball_track["frame"] >= start) & (ball_track["frame"] <= end) & ball_track["x"].notna() & ball_track["y"].notna()]
    if len(nearby) == 0:
        return None
    row = nearby.iloc[(nearby["frame"] - frame).abs().argmin()]
    return (float(row["x"]), float(row["y"]))


def shooter_assignment_confidence(distance_px: float | None) -> float:
    if distance_px is None or not np.isfinite(distance_px):
        return 0.0
    if distance_px <= 45:
        return 0.90
    if distance_px <= 85:
        return 0.70
    if distance_px <= MAX_SHOOTER_DISTANCE_PX:
        return 0.45
    return 0.20


def assign_likely_shooter(candidate: dict) -> dict:
    release_frame = int(candidate["release_frame"])
    best = None
    for f in range(max(0, release_frame - SHOOTER_LOOKBACK_FRAMES), release_frame + 1):
        ball_xy = ball_xy_at(f)
        if ball_xy is None:
            continue
        for d in detections_by_frame.get(f, []):
            if int(d["class_id"]) != CLASS_PLAYER:
                continue
            anchor = bbox_upper_body_center(d["bbox_xyxy"])
            dist = euclidean(ball_xy, anchor)
            temporal_penalty = (release_frame - f) * 2.0
            score = dist + temporal_penalty
            if best is None or score < best["score"]:
                best = {
                    "score": score,
                    "frame": f,
                    "track_id": d.get("track_id"),
                    "distance_px": float(dist),
                    "bbox": d["bbox_xyxy"],
                }

    if best is None or best["distance_px"] > MAX_SHOOTER_DISTANCE_PX:
        return {
            "shooter_track_id": None,
            "shooter_distance_px": None,
            "shooter_assignment_confidence": 0.0,
            "shooter_frame": None,
            "shooter_bbox_xyxy": None,
        }

    return {
        "shooter_track_id": int(best["track_id"]) if best["track_id"] is not None else None,
        "shooter_distance_px": best["distance_px"],
        "shooter_assignment_confidence": shooter_assignment_confidence(best["distance_px"]),
        "shooter_frame": int(best["frame"]),
        "shooter_bbox_xyxy": best["bbox"],
    }


def scale_box_xyxy(box, factor: float = 0.5):
    x1, y1, x2, y2 = map(float, box)
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w, h = (x2 - x1) * factor, (y2 - y1) * factor
    return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def crop_box(frame: np.ndarray, box) -> np.ndarray | None:
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, np.round(box))
    x1, x2 = np.clip([x1, x2], 0, w - 1)
    y1, y2 = np.clip([y1, y2], 0, h - 1)
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2].copy()


def read_frame_at(video_path: Path, frame_idx: int) -> np.ndarray | None:
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None


def fit_team_classifier_from_video() -> object | None:
    if not ENABLE_TEAM_CLASSIFIER or TeamClassifier is None:
        print("Team classifier disabled or unavailable; team_id will be None.")
        return None

    cap = cv2.VideoCapture(str(SOURCE_VIDEO_PATH))
    crops = []
    for frame_idx in tqdm(range(0, FRAME_COUNT, TEAM_FIT_STRIDE), desc="collecting jersey crops"):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            continue
        players = [d for d in detections_by_frame.get(frame_idx, []) if int(d["class_id"]) == CLASS_PLAYER]
        for d in players:
            crop = crop_box(frame, scale_box_xyxy(d["bbox_xyxy"], factor=0.5))
            if crop is not None and crop.size:
                crops.append(crop)
                if len(crops) >= MAX_TEAM_FIT_CROPS:
                    break
        if len(crops) >= MAX_TEAM_FIT_CROPS:
            break
    cap.release()

    print(f"Collected {len(crops)} player crops for team fitting.")
    if len(crops) < MIN_TEAM_FIT_CROPS:
        print("Not enough crops for reliable team clustering; team_id will be None.")
        return None

    classifier = TeamClassifier(device="cuda" if torch.cuda.is_available() else "cpu")
    classifier.fit(crops)
    print("Team classifier fitted.")
    return classifier


def assign_team_to_shots(events: list[dict], classifier) -> list[dict]:
    for event in events:
        event["team_id"] = None
    if classifier is None:
        return events

    frame_cache = {}
    for event in events:
        bbox = event.get("shooter_bbox_xyxy")
        frame_idx = event.get("shooter_frame") or event.get("release_frame")
        if bbox is None or frame_idx is None:
            continue
        if frame_idx not in frame_cache:
            frame_cache[frame_idx] = read_frame_at(SOURCE_VIDEO_PATH, frame_idx)
        frame = frame_cache[frame_idx]
        if frame is None:
            continue
        crop = crop_box(frame, scale_box_xyxy(bbox, factor=0.5))
        if crop is None or crop.size == 0:
            continue
        try:
            event["team_id"] = int(classifier.predict([crop])[0])
        except Exception as exc:
            event["notes"] = (event.get("notes") or "") + f" team prediction failed: {exc};"
    return events


for event in shot_events:
    event.update(assign_likely_shooter(event))

team_classifier = fit_team_classifier_from_video()
shot_events = assign_team_to_shots(shot_events, team_classifier)

display(pd.DataFrame(shot_events)[[
    "shot_id", "release_frame", "result", "shooter_track_id", "shooter_distance_px", "shooter_assignment_confidence", "team_id"
]] if shot_events else pd.DataFrame())


collecting jersey crops:   0%|          | 0/8 [00:00<?, ?it/s]

collecting jersey crops:  25%|██▌       | 2/8 [00:00<00:00,  6.35it/s]

collecting jersey crops:  38%|███▊      | 3/8 [00:00<00:00,  5.51it/s]

collecting jersey crops:  50%|█████     | 4/8 [00:00<00:00,  5.04it/s]

collecting jersey crops:  62%|██████▎   | 5/8 [00:00<00:00,  4.79it/s]

...

Embedding extraction: 3it [00:01,  2.01it/s]

Team classifier fitted.


Embedding extraction: 0it [00:00, ?it/s]

Embedding extraction: 1it [00:00, 17.47it/s]

,shot_id,release_frame,result,shooter_track_id,shooter_distance_px,shooter_assignment_confidence,team_id
0,1,137,make,11,89.657787,0.45,1


## 11. Export shot events

Events are exported to JSONL for frame-accurate downstream processing and CSV for quick spreadsheet review. Empty values are written cleanly as `None`/blank values.


In [13]:
SHOT_EVENT_FIELDS = [
    "shot_id",
    "start_frame",
    "release_frame",
    "rim_frame",
    "end_frame",
    "start_time_sec",
    "release_time_sec",
    "rim_time_sec",
    "end_time_sec",
    "result",
    "result_confidence",
    "result_reason",
    "rim_x",
    "rim_y",
    "min_rim_distance_px",
    "ball_in_basket_frames",
    "shooter_track_id",
    "shooter_distance_px",
    "shooter_assignment_confidence",
    "team_id",
    "shot_x_court",
    "shot_y_court",
    "shot_x_image",
    "shot_y_image",
    "notes",
]


def pythonize(value):
    if value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, np.ndarray):
        return [pythonize(v) for v in value.tolist()]
    if isinstance(value, list):
        return [pythonize(v) for v in value]
    if not isinstance(value, (dict, tuple)):
        try:
            if bool(pd.isna(value)):
                return None
        except (TypeError, ValueError):
            pass
    return value


def event_with_export_fields(event: dict) -> dict:
    row = dict(event)
    for key in ["start_frame", "release_frame", "rim_frame", "end_frame"]:
        frame = row.get(key)
        row[key.replace("frame", "time_sec")] = (float(frame) / SOURCE_FPS) if frame is not None else None

    rim = rim_row_at(row.get("rim_frame", -1))
    row["rim_x"] = float(rim["rim_x"]) if rim is not None else None
    row["rim_y"] = float(rim["rim_y"]) if rim is not None else None
    row["notes"] = (row.get("notes") or "").strip()

    export_row = {field: pythonize(row.get(field)) for field in SHOT_EVENT_FIELDS}
    return export_row


def export_shot_events(events: list[dict], jsonl_path: Path, csv_path: Path):
    rows = [event_with_export_fields(e) for e in events]
    with jsonl_path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    csv_rows = []
    for row in rows:
        csv_row = row.copy()
        if isinstance(csv_row.get("ball_in_basket_frames"), list):
            csv_row["ball_in_basket_frames"] = ";".join(map(str, csv_row["ball_in_basket_frames"]))
        csv_rows.append(csv_row)
    pd.DataFrame(csv_rows, columns=SHOT_EVENT_FIELDS).to_csv(csv_path, index=False)
    return rows


exported_events = export_shot_events(shot_events, SHOT_EVENTS_JSONL, SHOT_EVENTS_CSV)
print(f"Wrote JSONL: {SHOT_EVENTS_JSONL}")
print(f"Wrote CSV:   {SHOT_EVENTS_CSV}")
display(pd.DataFrame(exported_events))


Wrote JSONL: /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.jsonl
Wrote CSV:   /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.csv


,shot_id,start_frame,release_frame,rim_frame,end_frame,start_time_sec,release_time_sec,rim_time_sec,end_time_sec,result,...,ball_in_basket_frames,shooter_track_id,shooter_distance_px,shooter_assignment_confidence,team_id,shot_x_court,shot_y_court,shot_x_image,shot_y_image,notes
0,1,137,137,219,237,4.566667,4.566667,7.3,7.9,make,...,"[220, 221, 222, 223, 224, 225, 226, 227, 228, ...",11,89.657787,0.45,1,80.235191,44.414345,1084.78479,897.082275,


## 12. Render annotated shot-tracking video

The annotated video overlays player/rim/ball detections, the smoothed ball trail, the hoop zone, and active shot events. OpenCV writes the working MP4; `imageio-ffmpeg` re-encodes to H.264 for reliable notebook playback.


In [14]:
BOX_COLORS = {
    CLASS_BALL: (0, 255, 255),
    CLASS_BALL_IN_BASKET: (0, 220, 0),
    CLASS_PLAYER: (255, 180, 40),
    CLASS_REFEREE: (180, 180, 180),
    CLASS_RIM: (40, 80, 255),
    CLASS_NUMBER: (200, 120, 255),
}
RESULT_COLORS = {
    "make": (40, 210, 70),
    "miss": (40, 80, 255),
    "unknown": (180, 180, 180),
}
TRAIL_LENGTH = 36


def draw_label(frame, text: str, xy: tuple[int, int], color=(255, 255, 255), bg=(0, 0, 0), scale=0.5):
    x, y = xy
    (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    cv2.rectangle(frame, (x, y - th - baseline - 4), (x + tw + 6, y + 4), bg, -1)
    cv2.putText(frame, text, (x + 3, y - baseline - 1), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)


def draw_box(frame, bbox, label: str, color):
    x1, y1, x2, y2 = map(int, np.round(bbox))
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    draw_label(frame, label, (x1, max(18, y1)), color=(255, 255, 255), bg=color, scale=0.45)


def active_events_for_frame(events: list[dict], frame_idx: int) -> list[dict]:
    active = []
    for event in events:
        start = int(event.get("start_frame", -1))
        end = int(event.get("end_frame", -1))
        if start <= frame_idx <= end:
            active.append(event)
        elif end < frame_idx <= end + 75:
            active.append(event)
    return active


def draw_ball_trail(frame, frame_idx: int):
    trail = ball_track[
        (ball_track["frame"] >= max(0, frame_idx - TRAIL_LENGTH))
        & (ball_track["frame"] <= frame_idx)
        & ball_track["x"].notna()
        & ball_track["y"].notna()
    ]
    pts = [(int(r["x"]), int(r["y"])) for _, r in trail.iterrows()]
    for i in range(1, len(pts)):
        alpha = i / max(1, len(pts) - 1)
        color = (0, int(180 + 75 * alpha), 255)
        cv2.line(frame, pts[i - 1], pts[i], color, 2)
    if pts:
        cv2.circle(frame, pts[-1], 5, (0, 255, 255), -1)


def draw_hoop_zone(frame, frame_idx: int):
    zone = hoop_zone(frame_idx)
    if zone is None:
        return
    rim_x, rim_y, width, height = zone
    x1, y1 = int(rim_x - width / 2), int(rim_y - height / 2)
    x2, y2 = int(rim_x + width / 2), int(rim_y + height / 2)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (80, 220, 255), 1)
    cv2.circle(frame, (int(rim_x), int(rim_y)), 4, (80, 220, 255), -1)


def annotate_shot_frame(frame: np.ndarray, frame_idx: int, events: list[dict]) -> np.ndarray:
    annotated = frame.copy()
    dets = detections_by_frame.get(frame_idx, [])

    for d in dets:
        cid = int(d["class_id"])
        if cid not in [CLASS_PLAYER, CLASS_RIM, CLASS_BALL, CLASS_BALL_IN_BASKET]:
            continue
        color = BOX_COLORS.get(cid, (255, 255, 255))
        label = CLASS_NAMES.get(cid, str(cid))
        if d.get("track_id") is not None:
            label += f"#{d['track_id']}"
        if cid in [CLASS_BALL, CLASS_BALL_IN_BASKET, CLASS_RIM]:
            label += f" {float(d.get('conf', 0.0)):.2f}"
        draw_box(annotated, d["bbox_xyxy"], label, color)

    draw_ball_trail(annotated, frame_idx)
    draw_hoop_zone(annotated, frame_idx)

    overlay_lines = []
    for event in active_events_for_frame(events, frame_idx):
        result = event.get("result", "unknown")
        line = f"Shot {event['shot_id']}: {result.upper()} ({event.get('result_confidence', 0):.2f})"
        if event.get("shooter_track_id") is not None:
            line += f"  shooter #{event['shooter_track_id']}"
        if event.get("team_id") is not None:
            line += f"  team {event['team_id']}"
        overlay_lines.append((line, RESULT_COLORS.get(result, (180, 180, 180))))
        if frame_idx == event.get("release_frame"):
            xy = ball_xy_at(frame_idx)
            if xy is not None:
                cv2.circle(annotated, (int(xy[0]), int(xy[1])), 12, (255, 255, 255), 2)
                draw_label(annotated, "release", (int(xy[0]) + 8, int(xy[1]) - 8), bg=(40, 40, 40))

    y = 30
    for line, color in overlay_lines[:5]:
        draw_label(annotated, line, (18, y), bg=color, scale=0.58)
        y += 28

    cv2.putText(
        annotated,
        f"frame {frame_idx}",
        (18, annotated.shape[0] - 18),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )
    return annotated


def render_shot_video(events: list[dict], target_path: Path) -> Path:
    source_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
    cap = cv2.VideoCapture(str(SOURCE_VIDEO_PATH))
    with sv.VideoSink(str(target_path), source_info) as sink:
        for frame_idx in tqdm(range(FRAME_COUNT), desc="writing shot video"):
            ok, frame = cap.read()
            if not ok:
                break
            sink.write_frame(annotate_shot_frame(frame, frame_idx, events))
    cap.release()
    return target_path


def compress_video_h264(source_path: Path, target_path: Path, crf: int = 28) -> Path:
    cmd = [
        FFMPEG,
        "-y",
        "-loglevel",
        "error",
        "-i",
        str(source_path),
        "-vcodec",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-crf",
        str(crf),
        str(target_path),
    ]
    subprocess.run(cmd, check=True)
    return target_path


render_shot_video(shot_events, TARGET_VIDEO_PATH)
written_frames = count_decodable_frames(TARGET_VIDEO_PATH)
print(f"Wrote: {TARGET_VIDEO_PATH}")
print(f"Expected frames: {SOURCE_FRAME_COUNT}")
print(f"Output frames:   {written_frames}")
if written_frames < SOURCE_FRAME_COUNT * 0.95:
    warnings.warn(f"Annotated video may be truncated: {written_frames}/{SOURCE_FRAME_COUNT} frames")

compress_video_h264(TARGET_VIDEO_PATH, TARGET_VIDEO_COMPRESSED_PATH)
compressed_frames = count_decodable_frames(TARGET_VIDEO_COMPRESSED_PATH)
print(f"Wrote: {TARGET_VIDEO_COMPRESSED_PATH}")
print(f"Size:  {TARGET_VIDEO_COMPRESSED_PATH.stat().st_size/1e6:.1f} MB")
print(f"Frames: {compressed_frames}")


writing shot video:   0%|          | 0/238 [00:00<?, ?it/s]

writing shot video:   1%|          | 2/238 [00:00<00:12, 18.69it/s]

writing shot video:   3%|▎         | 6/238 [00:00<00:07, 29.31it/s]

writing shot video:   4%|▍         | 10/238 [00:00<00:07, 32.24it/s]

writing shot video:   6%|▌         | 14/238 [00:00<00:06, 35.08it/s]

...

writing shot video:  98%|█████████▊| 233/238 [00:06<00:00, 37.77it/s]

writing shot video: 100%|█████████▉| 237/238 [00:06<00:00, 38.36it/s]

writing shot video: 100%|██████████| 238/238 [00:06<00:00, 38.55it/s]

Wrote: /home/sagemaker-user/DLCNN_A3/runs_ilias/videos/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shots.mp4
Expected frames: 238
Output frames:   238


Wrote: /home/sagemaker-user/DLCNN_A3/runs_ilias/compressed/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shots-compressed.mp4
Size:  4.8 MB
Frames: 238


## 13. Render shot chart / court map

When court coordinates are available, this draws makes, misses, and unknown shots on an NBA court diagram. If CVM projection is unavailable, the notebook still saves an image-space fallback chart so the run has a visual shot summary.


In [15]:
def render_court_shot_chart(events: list[dict], path: Path) -> bool:
    valid_events = [e for e in events if e.get("shot_x_court") is not None and e.get("shot_y_court") is not None]
    if not valid_events or not SPORTS_AVAILABLE or court_config is None or draw_court is None:
        return False

    court = draw_court(config=court_config)
    result_to_color = {
        "make": sv.Color.GREEN,
        "miss": sv.Color.RED,
        "unknown": sv.Color.from_hex("#64748b"),
    }
    for result, color in result_to_color.items():
        xy = np.array([[e["shot_x_court"], e["shot_y_court"]] for e in valid_events if e.get("result") == result], dtype=float)
        if len(xy):
            court = draw_points_on_court(config=court_config, xy=xy, fill_color=color, court=court)

    cv2.imwrite(str(path), court)
    return True


def render_image_space_shot_chart(events: list[dict], path: Path):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_facecolor("#111827")
    ax.set_title("Shot chart (image coordinates)")
    ax.set_xlim(0, SOURCE_META["width"] or 1920)
    ax.set_ylim(SOURCE_META["height"] or 1080, 0)
    ax.set_xlabel("x pixels")
    ax.set_ylabel("y pixels")
    colors = {"make": "#22c55e", "miss": "#ef4444", "unknown": "#94a3b8"}

    for result, color in colors.items():
        xs = [e.get("shot_x_image") for e in events if e.get("result") == result and e.get("shot_x_image") is not None]
        ys = [e.get("shot_y_image") for e in events if e.get("result") == result and e.get("shot_y_image") is not None]
        if xs:
            ax.scatter(xs, ys, label=result, c=color, s=80, edgecolors="white", linewidths=0.8)

    for e in events:
        if e.get("shot_x_image") is not None and e.get("shot_y_image") is not None:
            ax.text(e["shot_x_image"] + 6, e["shot_y_image"] - 6, str(e["shot_id"]), color="white", fontsize=9)

    ax.legend(loc="upper right")
    ax.grid(color="white", alpha=0.12)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


if render_court_shot_chart(shot_events, SHOT_CHART_PATH):
    print(f"Wrote court shot chart: {SHOT_CHART_PATH}")
else:
    render_image_space_shot_chart(shot_events, SHOT_CHART_PATH)
    print(f"Wrote image-space shot chart: {SHOT_CHART_PATH}")


Wrote court shot chart: /home/sagemaker-user/DLCNN_A3/runs_ilias/charts/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-chart.png


## 14. Summary statistics and validation

In [16]:
def save_ball_trajectory_plot(path: Path):
    fig, ax = plt.subplots(figsize=(12, 5))
    valid = ball_track[ball_track["x"].notna() & ball_track["y"].notna()]
    ax.plot(valid["frame"], valid["x"], label="ball x", linewidth=1.5)
    ax.plot(valid["frame"], valid["y"], label="ball y", linewidth=1.5)
    outliers = ball_track[ball_track["is_outlier"]]
    if len(outliers):
        ax.scatter(outliers["frame"], outliers["raw_y"], c="red", s=18, label="rejected outlier")
    for e in shot_events:
        ax.axvline(e["release_frame"], color="#2563eb", alpha=0.35, linestyle="--")
        ax.axvline(e["rim_frame"], color="#dc2626", alpha=0.35, linestyle=":")
    ax.set_title("Smoothed ball trajectory")
    ax.set_xlabel("frame")
    ax.set_ylabel("pixels")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_ball_rim_distance_plot(path: Path):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    valid = distance_df[distance_df["rim_distance_px"].notna()]
    ax.plot(valid["frame"], valid["rim_distance_px"], label="ball-rim distance", linewidth=1.5)
    ax.axhline(RIM_PROXIMITY_PX, color="red", linestyle="--", alpha=0.55, label="rim proximity threshold")
    for e in shot_events:
        color = {"make": "#16a34a", "miss": "#dc2626", "unknown": "#64748b"}.get(e.get("result"), "#64748b")
        ax.axvline(e["rim_frame"], color=color, alpha=0.6)
        ax.text(e["rim_frame"], RIM_PROXIMITY_PX, f" {e['shot_id']} {e['result']}", rotation=90, va="bottom", fontsize=8)
    ax.set_title("Ball-to-rim distance")
    ax.set_xlabel("frame")
    ax.set_ylabel("pixels")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_debug_sample_frames(events: list[dict], max_frames: int = 12):
    frames = {0, min(FRAME_COUNT - 1, FRAME_COUNT // 2)}
    for e in events:
        frames.update([e.get("start_frame"), e.get("release_frame"), e.get("rim_frame"), e.get("end_frame")])
    frames = [int(f) for f in sorted(frames) if f is not None and 0 <= int(f) < FRAME_COUNT][:max_frames]

    for f in frames:
        frame = read_frame_at(SOURCE_VIDEO_PATH, f)
        if frame is None:
            continue
        annotated = annotate_shot_frame(frame, f, events)
        out_path = DEBUG_DIR / f"{VIDEO_STEM}-debug-frame-{f:06d}.jpg"
        cv2.imwrite(str(out_path), annotated)
    print(f"Wrote {len(frames)} debug frames to {DEBUG_DIR}")


def print_summary(events: list[dict]):
    total = len(events)
    makes = sum(e.get("result") == "make" for e in events)
    misses = sum(e.get("result") == "miss" for e in events)
    unknown = sum(e.get("result") == "unknown" for e in events)
    known = makes + misses
    pct = (makes / known * 100.0) if known else None
    avg_conf = float(np.mean([e.get("result_confidence", 0.0) for e in events])) if events else 0.0
    assigned = sum(e.get("shooter_track_id") is not None for e in events)
    with_court = sum(e.get("shot_x_court") is not None for e in events)

    print("Shot tracking summary")
    print(f"  Total shots:                         {total}")
    print(f"  Makes:                               {makes}")
    print(f"  Misses:                              {misses}")
    print(f"  Unknown:                             {unknown}")
    print(f"  Shooting percentage excl. unknown:   {pct:.1f}%" if pct is not None else "  Shooting percentage excl. unknown:   n/a")
    print(f"  Average result confidence:           {avg_conf:.2f}")
    print(f"  Shots with assigned shooter:         {assigned}")
    print(f"  Shots with court coordinates:        {with_court}")
    print("\nOutput locations")
    print(f"  Detection log:   {DETECTIONS_JSONL}")
    print(f"  Events JSONL:    {SHOT_EVENTS_JSONL}")
    print(f"  Events CSV:      {SHOT_EVENTS_CSV}")
    print(f"  Video:           {TARGET_VIDEO_PATH}")
    print(f"  H.264 video:     {TARGET_VIDEO_COMPRESSED_PATH}")
    print(f"  Shot chart:      {SHOT_CHART_PATH}")
    print(f"  Debug frames:    {DEBUG_DIR}")
    print(f"  Trajectory plot: {BALL_TRAJECTORY_PLOT}")
    print(f"  Distance plot:   {BALL_RIM_DISTANCE_PLOT}")


save_ball_trajectory_plot(BALL_TRAJECTORY_PLOT)
save_ball_rim_distance_plot(BALL_RIM_DISTANCE_PLOT)
save_debug_sample_frames(shot_events)
print_summary(shot_events)

assert SHOT_EVENTS_JSONL.exists(), "Missing shot-events JSONL export"
assert SHOT_EVENTS_CSV.exists(), "Missing shot-events CSV export"
assert TARGET_VIDEO_PATH.exists(), "Missing annotated shot video"
assert TARGET_VIDEO_COMPRESSED_PATH.exists(), "Missing compressed shot video"
assert SHOT_CHART_PATH.exists(), "Missing shot chart"


Wrote 5 debug frames to /home/sagemaker-user/DLCNN_A3/runs_ilias/debug_frames
Shot tracking summary
  Total shots:                         1
  Makes:                               1
  Misses:                              0
  Unknown:                             0
  Shooting percentage excl. unknown:   100.0%
  Average result confidence:           0.92
  Shots with assigned shooter:         1
  Shots with court coordinates:        1

Output locations
  Detection log:   /home/sagemaker-user/DLCNN_A3/runs_ilias/detections/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-detections.jsonl
  Events JSONL:    /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.jsonl
  Events CSV:      /home/sagemaker-user/DLCNN_A3/runs_ilias/exports/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shot-events.csv
  Video:           /home/sagemaker-user/DLCNN_A3/runs_ilias/videos/boston-celtics-new-york-knicks-game-1-q1-04.28-04.20-shots.mp4